In [19]:
import pandas as pd

In [20]:
df = pd.read_csv("./clean_data.csv")

In [21]:
df.head(4)

,movieId,original_title,genres,actors,directors,overview,original_language,poster_path,avg_rating,total_rating_users,popularity_score,tmdbId,decade
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,John Lasseter,"Led by Woody, Andy's toys live happily in his ...",en,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...,3.92,215,21.076092,862,90s
1,2,Jumanji,Adventure|Children|Fantasy,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,Joe Johnston,When siblings Judy and Peter discover an encha...,en,https://image.tmdb.org/t/p/w500/vgpXmVaVyUL7GG...,3.43,110,16.162251,8844,90s
2,3,Grumpier Old Men,Comedy|Romance,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,Howard Deutch,A family wedding reignites the ancient feud be...,en,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...,3.26,52,12.941625,15602,90s
3,4,Waiting to Exhale,Comedy|Drama|Romance,Whitney Houston|Angela Bassett|Loretta Devine|...,Forest Whitaker,"Cheated on, mistreated and stepped on, the wom...",en,https://image.tmdb.org/t/p/w500/qJU6rfil5xLVb5...,2.36,7,4.901541,31357,90s


In [22]:
# df_filtered.directors.value_counts()


In [ ]:
df_director = df[df["total_rating_users"] >= 30]
director_counts = df_director["directors"].value_counts().reset_index()
director_counts.columns = ["director", "movie_count"]

avg_ratings = df_director.groupby("directors")["avg_rating"].mean().reset_index()
avg_ratings.columns = ["director", "avg_rating"]

summary = pd.merge(director_counts, avg_ratings, on="director")

summary = summary.sort_values(by="avg_rating", ascending=False)

summary = summary[summary.movie_count > 1]


# summary.to_csv("director_average_ratings.csv", index=False)
summary.sort_values(by="movie_count", ascending=False, inplace=True)

summary[summary['director'] == 'Jon Favreau']

,director,movie_count,avg_rating
0,Steven Spielberg,17,3.638235
1,Tim Burton,11,3.378182
2,David Fincher,10,3.650000
4,Martin Scorsese,9,3.968889
3,Ridley Scott,9,3.761111
...,...,...,...
125,Stephen Norrington,2,2.990000
159,Simon West,2,2.965000
121,Frank Marshall,2,2.845000
112,Paul W. S. Anderson,2,2.810000


In [24]:
summary

,director,movie_count,avg_rating
141,Frank Darabont,2,4.290000
170,David Lean,2,4.210000
66,Roman Polanski,3,4.163333
68,Sergio Leone,3,4.080000
169,John Huston,2,4.070000
...,...,...,...
125,Stephen Norrington,2,2.990000
159,Simon West,2,2.965000
121,Frank Marshall,2,2.845000
112,Paul W. S. Anderson,2,2.810000


In [25]:
df_filtered.actors.reset_index()

NameError: name 'df_filtered' is not defined

In [ ]:
actors_df = (
    df.assign(actor=df["actors"].str.split("|"))
      .explode("actor")
      .drop(columns=["actors"])
)

actor_summary = (
    actors_df.groupby("actor", as_index=False)
    .agg(
        movies_count=("movieId", "count"),
        total_users=("total_rating_users", "sum"),
        avg_rating=("avg_rating", "mean")
    )
    .sort_values(by="movies_count", ascending=False)
)

# print(actor_summary.head(10))
# actor_summary[actor_summary.actor == 'Jackie Chan']
actor_summary_filtered = actor_summary[
    (actor_summary['total_users'] > 20) &
    (actor_summary['movies_count'] > 5)
].reset_index()

# actor_summary_filtered[actor_summary_filtered.actor == 'Robert Downey Jr.']
actor_summary_filtered.to_csv()

,index,actor,movies_count,total_users,avg_rating
0,14692,Robert De Niro,62,1515,3.320161
1,15387,Samuel L. Jackson,59,1298,3.129661
2,2325,Bruce Willis,56,1872,2.934107
3,13002,Nicolas Cage,51,949,3.018824
4,8684,Johnny Depp,49,1268,3.272653
...,...,...,...,...,...
1643,16776,Teri Polo,6,117,3.088333
1644,9346,Kate Mara,6,57,2.680000
1645,2059,Boris Karloff,6,30,3.750000
1646,6422,Harold Gould,6,43,3.298333


In [ ]:
df['first_actor'] = df['actors'].str.split('|').str[0]

# 2️⃣ Group and aggregate
actor_summary = (
    df.groupby('first_actor')
      .agg(
          movie_count=('movieId', 'count'),
          total_rating_users=('total_rating_users', 'sum'),
          avg_rating=('avg_rating', 'mean')
      )
      .sort_values('movie_count', ascending=False)
)

# 3️⃣ Optional: round avg_rating for readability
actor_summary['avg_rating'] = actor_summary['avg_rating'].round(2)
# actor_summary[actor_summary.first_actor == 'Jackie Chan']
actor_summary

,movie_count,total_rating_users,avg_rating
first_actor,,,
Jackie Chan,39,317,3.25
Nicolas Cage,39,656,2.97
Robert De Niro,37,767,3.35
Johnny Depp,37,1128,3.33
Clint Eastwood,35,533,3.38
...,...,...,...
Zana Briski,1,4,3.88
Zbigniew Zamachowski,1,20,4.03
Zgougou,1,1,1.00
